In [ ]:
# Projet : Prédiction du Score Metacritic d'un Film
# Notebook Jupyter couvrant toutes les phases du projet

# =============================================================================
# IMPORTS ET CONFIGURATION
# =============================================================================

# Web Scraping
import requests
from bs4 import BeautifulSoup
import time
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# Data Processing
import pandas as pd
import numpy as np
import re
import json
import os
from datetime import datetime
from urllib.parse import urljoin, urlparse

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import xgboost as xgb

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Utilities
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Créer les dossiers nécessaires
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports/figures', exist_ok=True)

print("✅ Environnement configuré avec succès!")

# PHASE 1: WEB SCRAPING IMDB

In [ ]:
class IMDbScraper:
    """
    Scraper robuste pour extraire les données des films depuis IMDb
    avec gestion des délais, erreurs et sauvegarde progressive
    """
    
    def __init__(self, delay_range=(1, 3)):
        self.session = requests.Session()
        self.delay_range = delay_range
        self.base_url = "https://www.imdb.com"
        self.setup_headers()
        
    def setup_headers(self):
        """Configuration des en-têtes avec rotation des User-Agents"""
        user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        ]
        self.session.headers.update({
            'User-Agent': random.choice(user_agents),
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Accept-Encoding': 'gzip, deflate',
            'Connection': 'keep-alive',
        })
    
    def random_delay(self):
        """Délai aléatoire entre les requêtes"""
        delay = random.uniform(*self.delay_range)
        time.sleep(delay)
    
    def get_movie_urls(self, num_pages=10, start_year=2010, end_year=2024):
        """
        Récupère les URLs des films populaires depuis IMDb
        """
        movie_urls = []
        
        for page in tqdm(range(1, num_pages + 1), desc="📡 Récupération des URLs"):
            start_index = 50 * (page - 1) + 1
            search_url = f"{self.base_url}/search/title/?title_type=feature&release_date={start_year}-01-01,{end_year}-12-31&sort=num_votes,desc&start={start_index}"
            
            try:
                response = self.session.get(search_url)
                response.raise_for_status()
                
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Trouver les liens des films
                movie_containers = soup.find_all('h3', class_='ipc-title')
                for container in movie_containers:
                    link = container.find('a')
                    if link and link.get('href'):
                        href = link.get('href')
                        full_url = urljoin(self.base_url, href)
                        movie_urls.append(full_url)
                
                self.random_delay()
                
            except Exception as e:
                print(f"❌ Erreur page {page}: {e}")
                continue
        
        return list(set(movie_urls))  # Supprimer les doublons
    
    def extract_movie_details(self, url):
        """
        Extrait tous les détails d'un film depuis sa page IMDb
        """
        try:
            response = self.session.get(url)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Structure des données
            movie_data = {
                'url': url,
                'title': None,
                'year': None,
                'duration': None,
                'imdb_rating': None,
                'num_votes': None,
                'genres': [],
                'director': None,
                'actors': [],
                'synopsis': None,
                'metacritic_score': None,
                'box_office': None,
                'budget': None,
                'content_rating': None,
                'country': None,
                'language': None
            }
            
            # Titre principal
            title_elem = soup.find('h1', {'data-testid': 'hero-title-block__title'})
            if title_elem:
                movie_data['title'] = title_elem.get_text().strip()
            
            # Année de sortie
            year_elem = soup.find('a', href=re.compile(r'/title/tt\d+/releaseinfo'))
            if year_elem:
                year_text = year_elem.get_text()
                year_match = re.search(r'(\d{4})', year_text)
                if year_match:
                    movie_data['year'] = int(year_match.group(1))
            
            # Durée
            duration_elem = soup.find('li', {'data-testid': 'title-techspec_runtime'})
            if duration_elem:
                duration_text = duration_elem.get_text()
                duration_match = re.search(r'(\d+)', duration_text)
                if duration_match:
                    movie_data['duration'] = int(duration_match.group(1))
            
            # Note IMDb
            rating_elem = soup.find('span', class_='sc-bde20123-1')
            if rating_elem:
                rating_text = rating_elem.get_text()
                try:
                    movie_data['imdb_rating'] = float(rating_text)
                except ValueError:
                    pass
            
            # Nombre de votes
            votes_elem = soup.find('div', class_='sc-bde20123-3')
            if votes_elem:
                votes_text = votes_elem.get_text()
                votes_match = re.search(r'([\d,]+)', votes_text)
                if votes_match:
                    movie_data['num_votes'] = int(votes_match.group(1).replace(',', ''))
            
            # Genres
            genres_section = soup.find('div', {'data-testid': 'genres'})
            if genres_section:
                genre_links = genres_section.find_all('a')
                movie_data['genres'] = [genre.get_text().strip() for genre in genre_links]
            
            # Réalisateur
            director_elem = soup.find('a', href=re.compile(r'/name/nm\d+'))
            if director_elem:
                movie_data['director'] = director_elem.get_text().strip()
            
            # Acteurs principaux
            cast_section = soup.find('section', {'data-testid': 'title-cast'})
            if cast_section:
                actor_links = cast_section.find_all('a', href=re.compile(r'/name/nm\d+'))[:5]
                movie_data['actors'] = [actor.get_text().strip() for actor in actor_links]
            
            # Synopsis
            synopsis_elem = soup.find('span', {'data-testid': 'plot-xl'})
            if synopsis_elem:
                movie_data['synopsis'] = synopsis_elem.get_text().strip()
            
            # Score Metacritic
            metacritic_elem = soup.find('span', class_='metacritic-score-box')
            if metacritic_elem:
                score_text = metacritic_elem.get_text().strip()
                try:
                    movie_data['metacritic_score'] = int(score_text)
                except ValueError:
                    pass
            
            # Classification du contenu
            rating_elem = soup.find('li', {'data-testid': 'title-techspec_certificate'})
            if rating_elem:
                movie_data['content_rating'] = rating_elem.get_text().strip()
            
            # Pays et langue
            country_elem = soup.find('li', {'data-testid': 'title-details-origin'})
            if country_elem:
                movie_data['country'] = country_elem.get_text().strip()
            
            language_elem = soup.find('li', {'data-testid': 'title-details-languages'})
            if language_elem:
                movie_data['language'] = language_elem.get_text().strip()
            
            return movie_data
            
        except Exception as e:
            print(f"❌ Erreur extraction {url}: {e}")
            return None
    
    def scrape_batch(self, urls, batch_size=30, save_interval=5):
        """
        Scrape par lots avec sauvegarde automatique
        """
        all_data = []
        
        for i in tqdm(range(0, len(urls), batch_size), desc="🎬 Scraping films"):
            batch_urls = urls[i:i+batch_size]
            batch_data = []
            
            for url in batch_urls:
                movie_data = self.extract_movie_details(url)
                if movie_data:
                    batch_data.append(movie_data)
                self.random_delay()
            
            all_data.extend(batch_data)
            
            # Sauvegarde intermédiaire
            if (i // batch_size + 1) % save_interval == 0:
                temp_df = pd.DataFrame(all_data)
                temp_df.to_csv(f'data/raw/movies_temp_{i//batch_size + 1}.csv', index=False)
                print(f"💾 Sauvegarde: {len(all_data)} films")
        
        return all_data

# Exécution du scraping
def run_scraping():
    """
    Fonction principale pour exécuter le scraping
    """
    print("🚀 Démarrage du scraping IMDb...")
    
    # Initialiser le scraper
    scraper = IMDbScraper()
    
    # Récupérer les URLs
    print("📡 Récupération des URLs des films...")
    movie_urls = scraper.get_movie_urls(num_pages=15, start_year=2010, end_year=2024)
    print(f"✅ {len(movie_urls)} URLs récupérées")
    
    # Sauvegarder les URLs
    with open('data/raw/movie_urls.json', 'w') as f:
        json.dump(movie_urls, f, indent=2)
    
    # Scraper les détails
    print("🎬 Scraping des détails des films...")
    movies_data = scraper.scrape_batch(movie_urls, batch_size=25, save_interval=3)
    
    # Sauvegarder le dataset final
    df = pd.DataFrame(movies_data)
    df.to_csv('data/raw/movies_raw.csv', index=False)
    
    print(f"✅ Scraping terminé! {len(df)} films dans le dataset")
    return df

# Exécuter le scraping (décommenter pour lancer)
# df_raw = run_scraping()

# Pour les tests, utiliser des données simulées
def create_sample_data():
    """Créer des données d'exemple pour les tests"""
    np.random.seed(42)
    
    genres_list = ['Action', 'Comedy', 'Drama', 'Thriller', 'Horror', 'Romance', 'Sci-Fi']
    directors = ['Christopher Nolan', 'Quentin Tarantino', 'Steven Spielberg', 'Martin Scorsese', 'Ridley Scott']
    
    sample_data = []
    for i in range(500):
        # Générer des scores corrélés mais avec du bruit
        imdb_base = np.random.normal(7.0, 1.5)
        metacritic_base = imdb_base * 10 + np.random.normal(0, 10)
        
        movie = {
            'title': f'Movie {i+1}',
            'year': np.random.randint(2010, 2024),
            'duration': np.random.randint(90, 180),
            'imdb_rating': max(1.0, min(10.0, imdb_base)),
            'num_votes': np.random.randint(1000, 500000),
            'genres': np.random.choice(genres_list, size=np.random.randint(1, 4), replace=False).tolist(),
            'director': np.random.choice(directors),
            'actors': [f'Actor {j}' for j in range(np.random.randint(3, 6))],
            'synopsis': f'This is a synopsis for movie {i+1}. ' * np.random.randint(5, 15),
            'metacritic_score': max(0, min(100, int(metacritic_base))),
            'box_office': np.random.randint(1000000, 500000000) if np.random.random() > 0.3 else None,
            'budget': np.random.randint(500000, 200000000) if np.random.random() > 0.4 else None,
            'content_rating': np.random.choice(['G', 'PG', 'PG-13', 'R']),
            'country': np.random.choice(['USA', 'UK', 'France', 'Germany']),
            'language': np.random.choice(['English', 'French', 'Spanish', 'German'])
        }
        sample_data.append(movie)
    
    return pd.DataFrame(sample_data)

# Créer les données d'exemple
df_raw = create_sample_data()
print(f"📊 Dataset d'exemple créé: {len(df_raw)} films")


# PHASE 2: NETTOYAGE ET PRÉTRAITEMENT

In [ ]:
def clean_data(df):
    """
    Nettoyage complet des données (version robuste)
    """
    print("🧹 Nettoyage des données...")
    
    df_clean = df.copy()
    
    # Supprimer les doublons
    df_clean = df_clean.drop_duplicates(subset=['title', 'year'])
    
    # Colonnes obligatoires avec valeurs par défaut si manquantes
    if 'genres' not in df_clean.columns:
        df_clean['genres'] = df_clean.get('genres', pd.Series([['Unknown']]*len(df_clean)))
    
    # Créer main_genre si elle n'existe pas
    if 'main_genre' not in df_clean.columns:
        df_clean['main_genre'] = df_clean['genres'].apply(lambda x: x[0] if x and isinstance(x, list) else 'Unknown')
    
    # Gestion des acteurs
    if 'actors' not in df_clean.columns:
        df_clean['actors'] = df_clean.get('actors', pd.Series([['Unknown']]*len(df_clean)))
    
    df_clean['num_actors'] = df_clean['actors'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    
    # Autres colonnes avec valeurs par défaut
    default_values = {
        'duration': df_clean['duration'].median() if 'duration' in df_clean.columns else 120,
        'num_votes': 0,
        'metacritic_score': 0,  # Valeur factice pour les nouvelles prédictions
        'imdb_rating': df_clean['imdb_rating'].mean() if 'imdb_rating' in df_clean.columns else 7.0,
        'year': datetime.now().year,
        'synopsis': '',
        'director': 'Unknown'
    }
    
    for col, default in default_values.items():
        if col not in df_clean.columns:
            df_clean[col] = default
    
    # Features dérivées
    df_clean['decade'] = (df_clean['year'] // 10) * 10
    df_clean['popularity_score'] = np.log1p(df_clean['num_votes'])
    df_clean['is_recent'] = (df_clean['year'] >= 2020).astype(int)
    df_clean['num_genres'] = df_clean['genres'].apply(len)
    
    print(f"✅ Nettoyage terminé: {len(df_clean)} films")
    return df_clean
def feature_engineering(df):
    """
    Feature engineering (version robuste)
    """
    print("🔧 Feature Engineering...")
    df_fe = df.copy()
    
    # Vérification et initialisation des colonnes manquantes
    expected_columns = {
        'budget': None,
        'box_office': None,
        'content_rating': 'Unknown',
        'country': 'Unknown',
        'language': 'Unknown',
        'director_experience': 1,
        'famous_director': 0,
        'avg_word_length': 0,
        'sentiment_score': 0,
        'word_count': 0
    }
    
    for col, default_val in expected_columns.items():
        if col not in df_fe.columns:
            df_fe[col] = default_val
    
    # Features du synopsis (NLP basique)
    def extract_synopsis_features(text):
        if not text:
            return 0, 0, 0
        
        # Longueur des mots
        words = text.split()
        avg_word_length = np.mean([len(word) for word in words]) if words else 0
        
        # Sentiment (basique)
        blob = TextBlob(text)
        sentiment = blob.sentiment.polarity
        
        # Nombre de mots
        word_count = len(words)
        
        return avg_word_length, sentiment, word_count
    
    synopsis_features = df_fe['synopsis'].apply(extract_synopsis_features)
    df_fe['avg_word_length'] = [x[0] for x in synopsis_features]
    df_fe['sentiment_score'] = [x[1] for x in synopsis_features]
    df_fe['word_count'] = [x[2] for x in synopsis_features]
    
    # Ratios financiers
    df_fe['roi'] = np.where(
        (df_fe['budget'].notna()) & (df_fe['box_office'].notna()) & (df_fe['budget'] > 0),
        df_fe['box_office'] / df_fe['budget'],
        np.nan
    )
    
    # Catégories de budget
    budget_median = df_fe['budget'].median()
    df_fe['budget_category'] = np.where(
        df_fe['budget'].isna(), 'Unknown',
        np.where(df_fe['budget'] < budget_median, 'Low', 'High')
    )
    
    print(f"✅ Feature Engineering terminé: {df_fe.shape[1]} features")
    return df_fe

# Appliquer le nettoyage et feature engineering
df_clean = clean_data(df_raw)
df_features = feature_engineering(df_clean)


# PHASE 3: ANALYSE EXPLORATOIRE (EDA)

In [ ]:
def explore_data(df):
    """
    Analyse exploratoire complète
    """
    print("📊 Analyse Exploratoire des Données...")
    
    # Informations générales
    print("\n" + "="*50)
    print("INFORMATIONS GÉNÉRALES")
    print("="*50)
    print(f"Nombre de films: {len(df)}")
    print(f"Période: {df['year'].min()} - {df['year'].max()}")
    print(f"Nombre de features: {df.shape[1]}")
    
    # Statistiques descriptives
    print("\n" + "="*50)
    print("STATISTIQUES DESCRIPTIVES")
    print("="*50)
    numeric_cols = ['imdb_rating', 'metacritic_score', 'duration', 'num_votes', 'year']
    print(df[numeric_cols].describe())
    
    # Valeurs manquantes
    print("\n" + "="*50)
    print("VALEURS MANQUANTES")
    print("="*50)
    missing_data = df.isnull().sum()
    missing_percent = (missing_data / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing_Count': missing_data,
        'Missing_Percentage': missing_percent
    })
    print(missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False))
    
    # Visualisations
    create_eda_plots(df)
    
    return df

def create_eda_plots(df):
    """
    Création des visualisations pour l'EDA (version robuste)
    """
    print("📈 Création des visualisations...")
    
    # Vérifier si main_genre existe, sinon créer une version factice
    if 'main_genre' not in df.columns:
        df['main_genre'] = 'Unknown'
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    try:
        # Distribution IMDb
        axes[0, 0].hist(df['imdb_rating'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        axes[0, 0].set_title('Distribution des Notes IMDb')
        
        # Distribution Metacritic
        axes[0, 1].hist(df['metacritic_score'], bins=30, alpha=0.7, color='lightcoral', edgecolor='black')
        axes[0, 1].set_title('Distribution des Scores Metacritic')
        
        # Relation IMDb vs Metacritic
        axes[1, 0].scatter(df['imdb_rating'], df['metacritic_score'], alpha=0.6, color='green')
        axes[1, 0].set_title('Relation IMDb vs Metacritic')
        
        # Distribution par décennie
        decade_counts = df['decade'].value_counts().sort_index()
        axes[1, 1].bar(decade_counts.index, decade_counts.values, color='orange', alpha=0.7)
        axes[1, 1].set_title('Nombre de Films par Décennie')
        
    except Exception as e:
        print(f"⚠️ Erreur lors de la création des graphiques: {e}")
    
    plt.tight_layout()
    plt.savefig('reports/figures/eda_distributions.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Deuxième série de graphiques
    plt.figure(figsize=(12, 8))
    
    try:
        # Top genres
        genre_counts = df['main_genre'].value_counts().head(10)
        plt.subplot(2, 2, 1)
        genre_counts.plot(kind='bar', color='steelblue')
        plt.title('Top 10 Genres')
        plt.xticks(rotation=45)
        
        # Scores par genre
        genre_scores = df.groupby('main_genre')['metacritic_score'].mean().sort_values(ascending=False).head(10)
        plt.subplot(2, 2, 2)
        genre_scores.plot(kind='bar', color='coral')
        plt.title('Score Metacritic Moyen par Genre')
        plt.xticks(rotation=45)
        
        # Évolution temporelle
        plt.subplot(2, 2, 3)
        yearly_scores = df.groupby('year')['metacritic_score'].mean()
        yearly_scores.plot(kind='line', marker='o', color='purple')
        plt.title('Évolution des Scores Metacritic par Année')
        
        # Durée vs Score
        plt.subplot(2, 2, 4)
        plt.scatter(df['duration'], df['metacritic_score'], alpha=0.6, color='brown')
        plt.title('Durée vs Score Metacritic')
        
    except Exception as e:
        print(f"⚠️ Erreur lors de la création des graphiques: {e}")
    
    plt.tight_layout()
    plt.savefig('reports/figures/eda_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
# Exécuter l'EDA
df_explored = explore_data(df_features)


# PHASE 4: MODÉLISATION MACHINE LEARNING

In [ ]:
def prepare_ml_data(df):
    """
    Préparer les données pour le machine learning (version robuste)
    """
    print("🤖 Préparation des données pour ML...")
    
    # Vérifier que la colonne cible existe
    if 'metacritic_score' not in df.columns:
        df['metacritic_score'] = 0  # Valeur factice si manquante
    
    # Liste complète des features attendues
    base_features = [
        'imdb_rating', 'duration', 'num_votes', 'year', 'num_genres',
        'popularity_score', 'is_recent', 'num_actors', 'director_experience',
        'famous_director', 'avg_word_length', 'sentiment_score', 'word_count'
    ]
    
    # Ajouter les genres encodés
    genre_columns = [col for col in df.columns if col.startswith('genre_')]
    feature_columns = base_features + genre_columns
    
    # Vérifier et créer les features manquantes
    for feat in base_features:
        if feat not in df.columns:
            if feat == 'num_genres':
                df['num_genres'] = df['genres'].apply(len)
            elif feat == 'popularity_score':
                df['popularity_score'] = np.log1p(df['num_votes'])
            elif feat == 'is_recent':
                df['is_recent'] = (df['year'] >= 2020).astype(int)
            elif feat not in df.columns:
                df[feat] = 0  # Valeur par défaut pour les autres features
    
    # Créer le dataset ML
    X = df[feature_columns].copy()
    y = df['metacritic_score'].copy()
    
    # Standardisation des features
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=X.columns,
        index=X.index
    )
    
    print(f"✅ Dataset ML préparé: {X.shape[0]} samples, {X.shape[1]} features")
    return X_scaled, y, scaler, feature_columns

def train_models(X, y):
    """
    Entraîner et évaluer plusieurs modèles
    """
    print("🚀 Entraînement des modèles...")
    
    # Split train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=pd.qcut(y, q=5, labels=False)
    )
    
    # Définir les modèles
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(alpha=1.0),
        'Lasso': Lasso(alpha=1.0),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
    }
    
    # Résultats
    results = {}
    trained_models = {}
    
    for name, model in models.items():
        print(f"📊 Entraînement {name}...")
        
        # Entraînement
        model.fit(X_train, y_train)
        
        # Prédictions
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        
        # Métriques
        train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
        train_mae = mean_absolute_error(y_train, y_pred_train)
        test_mae = mean_absolute_error(y_test, y_pred_test)
        train_r2 = r2_score(y_train, y_pred_train)
        test_r2 = r2_score(y_test, y_pred_test)
        
        # Cross-validation
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
        cv_rmse = np.sqrt(-cv_scores.mean())
        cv_std = np.sqrt(cv_scores.std())
        
        results[name] = {
            'Train RMSE': train_rmse,
            'Test RMSE': test_rmse,
            'Train MAE': train_mae,
            'Test MAE': test_mae,
            'Train R²': train_r2,
            'Test R²': test_r2,
            'CV RMSE': cv_rmse,
            'CV Std': cv_std,
            'Predictions': y_pred_test
        }
        
        trained_models[name] = model
    
    return results, trained_models, X_train, X_test, y_train, y_test

def evaluate_models(results):
    """
    Évaluer et comparer les modèles
    """
    print("\n" + "="*70)
    print("RÉSULTATS DES MODÈLES")
    print("="*70)
    
    # Créer DataFrame des résultats
    results_df = pd.DataFrame({
        name: {
            'Test RMSE': metrics['Test RMSE'],
            'Test MAE': metrics['Test MAE'],
            'Test R²': metrics['Test R²'],
            'CV RMSE': metrics['CV RMSE']
        }
        for name, metrics in results.items()
    }).T
    
    # Trier par performance
    results_df = results_df.sort_values('Test RMSE')
    
    print(results_df.round(4))
    
    # Meilleur modèle
    best_model = results_df.index[0]
    print(f"\n🏆 Meilleur modèle: {best_model}")
    print(f"   Test RMSE: {results_df.loc[best_model, 'Test RMSE']:.4f}")
    print(f"   Test R²: {results_df.loc[best_model, 'Test R²']:.4f}")
    
    return results_df, best_model

def feature_importance_analysis(models, feature_columns):
    """
    Analyser l'importance des features
    """
    print("\n📊 Analyse de l'importance des features...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    model_names = ['Ridge', 'Lasso', 'Random Forest', 'XGBoost']
    
    for i, name in enumerate(model_names):
        if name in models:
            model = models[name]
            
            if hasattr(model, 'feature_importances_'):
                # Tree-based models
                importance = model.feature_importances_
            elif hasattr(model, 'coef_'):
                # Linear models
                importance = np.abs(model.coef_)
            else:
                continue
            
            # Top 10 features
            feature_importance = pd.DataFrame({
                'feature': feature_columns,
                'importance': importance
            }).sort_values('importance', ascending=False).head(10)
            
            axes[i].barh(range(len(feature_importance)), feature_importance['importance'])
            axes[i].set_yticks(range(len(feature_importance)))
            axes[i].set_yticklabels(feature_importance['feature'])
            axes[i].set_title(f'Top 10 Features - {name}')
            axes[i].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('reports/figures/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

def create_prediction_plots(results, y_test):
    """
    Créer les graphiques de prédiction
    """
    print("📈 Création des graphiques de prédiction...")
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, (name, metrics) in enumerate(results.items()):
        if i >= 6:  # Limiter à 6 graphiques
            break
            
        y_pred = metrics['Predictions']
        
        # Scatter plot prédictions vs réalité
        axes[i].scatter(y_test, y_pred, alpha=0.6, color='blue')
        axes[i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[i].set_xlabel('Scores Réels')
        axes[i].set_ylabel('Scores Prédits')
        axes[i].set_title(f'{name}\nR² = {metrics["Test R²"]:.3f}')
        
        # Ajouter ligne de référence
        axes[i].grid(True, alpha=0.3)
    
    # Masquer les axes inutilisés
    for i in range(len(results), 6):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('reports/figures/predictions_vs_reality.png', dpi=300, bbox_inches='tight')
    plt.show()

def hyperparameter_tuning(X_train, y_train, X_test, y_test):
    """
    Optimisation des hyperparamètres pour les meilleurs modèles
    """
    print("🔧 Optimisation des hyperparamètres...")
    
    # Grilles de paramètres
    param_grids = {
        'Random Forest': {
            'n_estimators': [100, 200, 300],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        },
        'XGBoost': {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 6, 9],
            'learning_rate': [0.01, 0.1, 0.2],
            'subsample': [0.8, 0.9, 1.0]
        }
    }
    
    tuned_models = {}
    tuned_results = {}
    
    for name, param_grid in param_grids.items():
        print(f"🎯 Tuning {name}...")
        
        if name == 'Random Forest':
            base_model = RandomForestRegressor(random_state=42)
        elif name == 'XGBoost':
            base_model = xgb.XGBRegressor(random_state=42, verbosity=0)
        
        # Grid Search
        grid_search = GridSearchCV(
            base_model, param_grid, cv=5, scoring='neg_mean_squared_error',
            n_jobs=-1, verbose=0
        )
        
        grid_search.fit(X_train, y_train)
        
        # Meilleur modèle
        best_model = grid_search.best_estimator_
        
        # Évaluation
        y_pred_test = best_model.predict(X_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
        test_r2 = r2_score(y_test, y_pred_test)
        
        tuned_models[name] = best_model
        tuned_results[name] = {
            'Best Params': grid_search.best_params_,
            'Test RMSE': test_rmse,
            'Test R²': test_r2,
            'Predictions': y_pred_test
        }
        
        print(f"✅ {name} - RMSE: {test_rmse:.4f}, R²: {test_r2:.4f}")
    
    return tuned_models, tuned_results

def detect_over_under_rated(df, best_model, X_test, y_test):
    """
    Détecter les films surcotés/sous-cotés
    """
    print("🔍 Détection des films surcotés/sous-cotés...")
    
    # Prédictions
    y_pred = best_model.predict(X_test)
    
    # Calcul des résidus
    residuals = y_test - y_pred
    
    # Créer DataFrame des résultats
    test_indices = X_test.index
    analysis_df = df.loc[test_indices].copy()
    analysis_df['predicted_score'] = y_pred
    analysis_df['residual'] = residuals
    analysis_df['abs_residual'] = np.abs(residuals)
    
    # Seuils pour classification
    threshold = np.std(residuals) * 1.5
    
    analysis_df['rating_category'] = 'Normal'
    analysis_df.loc[residuals > threshold, 'rating_category'] = 'Sous-coté'
    analysis_df.loc[residuals < -threshold, 'rating_category'] = 'Surcôté'
    
    # Top films surcotés et sous-cotés
    overrated = analysis_df[analysis_df['rating_category'] == 'Surcôté'].nlargest(10, 'abs_residual')
    underrated = analysis_df[analysis_df['rating_category'] == 'Sous-coté'].nlargest(10, 'abs_residual')
    
    print("\n🔻 TOP 10 FILMS SURCOTÉS (Metacritic > Prédiction)")
    print(overrated[['title', 'year', 'metacritic_score', 'predicted_score', 'residual']].round(2))
    
    print("\n🔺 TOP 10 FILMS SOUS-COTÉS (Metacritic < Prédiction)")
    print(underrated[['title', 'year', 'metacritic_score', 'predicted_score', 'residual']].round(2))
    
    return analysis_df, overrated, underrated

def create_final_report(results_df, best_model_name, feature_columns, analysis_df):
    """
    Créer le rapport final
    """
    print("📋 Création du rapport final...")
    
    # Graphique final de comparaison
    plt.figure(figsize=(14, 10))
    
    # Subplot 1: Comparaison des modèles
    plt.subplot(2, 2, 1)
    results_df['Test RMSE'].plot(kind='bar', color='skyblue')
    plt.title('Comparaison RMSE des Modèles')
    plt.ylabel('RMSE')
    plt.xticks(rotation=45)
    
    # Subplot 2: Distribution des résidus
    plt.subplot(2, 2, 2)
    residuals = analysis_df['residual']
    plt.hist(residuals, bins=30, alpha=0.7, color='lightcoral')
    plt.axvline(0, color='red', linestyle='--', linewidth=2)
    plt.title('Distribution des Résidus')
    plt.xlabel('Résidus')
    plt.ylabel('Fréquence')
    
    # Subplot 3: Catégories de films
    plt.subplot(2, 2, 3)
    category_counts = analysis_df['rating_category'].value_counts()
    plt.pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%')
    plt.title('Répartition des Films')
    
    # Subplot 4: Résidus vs Prédictions
    plt.subplot(2, 2, 4)
    plt.scatter(analysis_df['predicted_score'], analysis_df['residual'], alpha=0.6)
    plt.axhline(0, color='red', linestyle='--', linewidth=2)
    plt.xlabel('Scores Prédits')
    plt.ylabel('Résidus')
    plt.title('Résidus vs Prédictions')
    
    plt.tight_layout()
    plt.savefig('reports/figures/final_report.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Statistiques finales
    print("\n" + "="*70)
    print("RAPPORT FINAL")
    print("="*70)
    print(f"🏆 Meilleur modèle: {best_model_name}")
    print(f"📊 Performance (RMSE): {results_df.loc[best_model_name, 'Test RMSE']:.4f}")
    print(f"📊 Performance (R²): {results_df.loc[best_model_name, 'Test R²']:.4f}")
    print(f"🎯 Nombre de features: {len(feature_columns)}")
    print(f"📈 Films analysés: {len(analysis_df)}")
    
    category_stats = analysis_df['rating_category'].value_counts()
    print(f"\n🔍 Répartition des prédictions:")
    for category, count in category_stats.items():
        percentage = (count / len(analysis_df)) * 100
        print(f"   {category}: {count} films ({percentage:.1f}%)")

# EXÉCUTION PRINCIPALE

In [ ]:
def main():
    """
    Fonction principale pour exécuter tout le pipeline ML
    """
    print("🚀 Démarrage du pipeline Machine Learning...")
    
    # Préparer les données
    X, y, scaler, feature_columns = prepare_ml_data(df_features)
    
    # Entraîner les modèles
    results, trained_models, X_train, X_test, y_train, y_test = train_models(X, y)
    
    # Évaluer les modèles
    results_df, best_model_name = evaluate_models(results)
    
    # Analyse de l'importance des features
    feature_importance_analysis(trained_models, feature_columns)
    
    # Graphiques de prédiction
    create_prediction_plots(results, y_test)
    
    # Optimisation des hyperparamètres
    tuned_models, tuned_results = hyperparameter_tuning(X_train, y_train, X_test, y_test)
    
    # Sélectionner le meilleur modèle (original ou tuné)
    best_model = trained_models[best_model_name]
    if best_model_name in tuned_results:
        if tuned_results[best_model_name]['Test RMSE'] < results[best_model_name]['Test RMSE']:
            best_model = tuned_models[best_model_name]
            print(f"🎯 Modèle optimisé sélectionné: {best_model_name}")
    
    # Détection des films surcotés/sous-cotés
    analysis_df, overrated, underrated = detect_over_under_rated(df_features, best_model, X_test, y_test)
    
    # Rapport final
    create_final_report(results_df, best_model_name, feature_columns, analysis_df)
    
    # Sauvegarder les résultats
    analysis_df.to_csv('data/processed/movie_analysis.csv', index=False)
    overrated.to_csv('data/processed/overrated_movies.csv', index=False)
    underrated.to_csv('data/processed/underrated_movies.csv', index=False)
    
    print("\n✅ Pipeline terminé avec succès!")
    print("📁 Fichiers sauvegardés:")
    print("   - data/processed/movie_analysis.csv")
    print("   - data/processed/overrated_movies.csv")
    print("   - data/processed/underrated_movies.csv")
    print("   - reports/figures/*.png")
    
    return best_model, analysis_df, results_df

# Exécuter le pipeline complet
if __name__ == "__main__":
    best_model, analysis_df, results_df = main()


# FONCTION DE PRÉDICTION POUR NOUVEAUX FILMS

In [ ]:
def predict_new_movie(title, year, imdb_rating, duration, genres, director, synopsis=""):
    """
    Prédire le score Metacritic pour un nouveau film (version finale)
    """
    print(f"🎬 Prédiction pour: {title} ({year})")
    
    # Créer un DataFrame minimal
    new_movie = pd.DataFrame({
        'title': [title],
        'year': [year],
        'imdb_rating': [imdb_rating],
        'duration': [duration],
        'genres': [genres],
        'director': [director],
        'synopsis': [synopsis]
    })
    
    # Appliquer le nettoyage qui va créer toutes les colonnes nécessaires
    new_movie = clean_data(new_movie)
    
    # Appliquer le feature engineering
    new_movie = feature_engineering(new_movie)
    
    # Préparer les données pour le modèle
    X_new, _, _, _ = prepare_ml_data(new_movie)
    
    # Faire la prédiction
    prediction = best_model.predict(X_new)[0]
    
    print(f"📊 Score Metacritic prédit: {prediction:.1f}")
    
    # Interprétation
    if prediction >= 80:
        category = "Excellent"
    elif prediction >= 60:
        category = "Bon"
    elif prediction >= 40:
        category = "Moyen"
    else:
        category = "Faible"
    
    print(f"🏆 Catégorie: {category}")
    return prediction
    
# Exemple d'utilisation
print("\n" + "="*70)
print("EXEMPLE DE PRÉDICTION")
print("="*70)

# Prédire pour un film fictif
predict_new_movie(
    title="The Dark Knight Returns",
    year=2024,
    imdb_rating=8.5,
    duration=150,
    genres=['Action', 'Drama', 'Crime'],
    director="Christopher Nolan",
    synopsis="The Caped Crusader battles crime in Gotham City."
)

print("\n🎉 Projet terminé avec succès!")